# ROGII — kernels-only submission (v5 ensemble, sacred 9.155)

Run **on Kaggle** (internet OFF) → writes `/kaggle/working/submission.csv` → **Submit to Competition**.
Attach: the competition data + **`rogii-code`** (repo `src/`) + **`rogii-models`** (probs/v5_lgb0, v5_cat3,
v5_cat4, v5_cat5 — each contains model_full.pkl). See docs/submission_workflow.md.

## 1. Code on path + data location

In [ ]:
import sys, os, glob, joblib
import numpy as np
sys.path.insert(0, "/kaggle/input/rogii-code")            # dataset root containing the `src` folder
os.environ["ROGII_DATA_DIR"] = "/kaggle/input/rogii-wellbore-geology-prediction"     # so src.data reads the comp data
from src import cv, data, kernel9251 as k9, submission     # noqa: E402
from src.config import TRAIN_DIR                            # noqa: E402
print("test wells:", data.list_well_ids("test"))


## 2. Build the 222 features for the test wells (imputers fit on the same dev split)

In [ ]:
dev, _ = cv.sacred_split(data.list_well_ids("train"))   # deterministic — matches training
k9.fit_imputers(dev, TRAIN_DIR)
test_df = k9.build_dataset([data.horizontal_path(w, "test") for w in data.list_well_ids("test")],
                           is_train=False, label="test_sub")
feats = [c for c in test_df.columns if c not in {"well", "id", "target"}]
Xt = test_df[feats].astype("float32")
anchor = test_df["last_known_tvt"].to_numpy(float)
print("test features:", Xt.shape)


## 3. Load the v5 models, blend, de-residualize → submission.csv

In [ ]:
WEIGHTS = {"v5_lgb0": 0.141, "v5_cat3": 0.114, "v5_cat4": 0.21, "v5_cat5": 0.535}
blend = np.zeros(len(Xt), dtype=float)
for name, w in WEIGHTS.items():
    hits = (glob.glob(f"/kaggle/input/rogii-models/**/{name}/model_full.pkl", recursive=True)
            or glob.glob(f"/kaggle/input/rogii-models/**/{name}.pkl", recursive=True))
    assert hits, f"model {name} not found under /kaggle/input/rogii-models"
    blend += w * joblib.load(hits[0]).predict(Xt)
tvt = blend + anchor                                        # de-residualize: drift + last-known TVT
ss = submission.build_submission(dict(zip(test_df["id"], tvt)))
ss.to_csv("/kaggle/working/submission.csv", index=False)
print("wrote submission.csv", ss.shape)
ss.head()


**Now click _Submit to Competition_.** Our sacred-holdout estimate is ~9.16; the 3-well LB is a separate noisy check — record it in the diary but keep trusting sacred for decisions.